In [11]:
import pandas_ta as pta
import numpy as np
import pandas as pd
import mplfinance as mpf
from tabulate import tabulate

from utils.KrakenHistoricalData import KrakenHistoricalData
from ggTrader.Signals import Signals
from ggTrader.Backtest import Backtest
import matplotlib.pyplot as plt
import matplotlib.ticker as tck  # Import the ticker module
import seaborn as sns



In [12]:

top20_kraken = [
    "BTC",  # 1 Bitcoin
    "ETH",  # 2 Ethereum
    "XRP",  # 3 XRP
    "BNB",  # 4 BNB
    "SOL",  # 5 Solana
    "TRX",  # 6 TRON
    "DOGE",  # 7 Dogecoin
    "ADA",  # 8 Cardano
    # "HYPE",  # 9 Hyperliquid
    "LINK",  # 10 Chainlink
    "BCH",  # 11 Bitcoin Cash
    "XLM",  # 12 Stellar
    "SUI",  # 13 Sui
    "HBAR",  # 14 Hedera
    "AVAX",  # 15 Avalanche
    "ZEC",  # 16 Zcash
    "LTC",  # 17 Litecoin
    "XMR",  # 18 Monero
    "SHIB",  # 19 Shiba Inu
    "TON",  # 20 Toncoin
    "CRO",  # 21 Crypto.com
    "DOT",  # 22 Polkadot
    "MNT",  # 23 Mantle
    "TAO",  # 24 Bittensor
    "UNI",  # 25 Uniswap
]

In [13]:
# Ticker

symbols = ["BTC"]
interval = "4h"

# Time Range

end = pd.to_datetime("2025-09-30").tz_localize('UTC')
start = end - pd.Timedelta(days=30 * 18)
# start = pd.to_datetime("2023-01-01").tz_localize('UTC')
# Indicator Params
sar_acceleration = 0.02
sar_maximum = 0.2
atr_multiplier = 3  #atr multiplier for chandelier exit
adx_threshold = 35  #trend strength
ce_high_length = 22  # look back length

# Fees
maker_fee = 0.0025
taker_fee = 0.004
transaction_fee = (maker_fee + taker_fee) / 2

# portfolio
# init_cash = 1000

#
k = KrakenHistoricalData()
s = Signals(

    atr_multiplier=atr_multiplier,
    adx_threshold=adx_threshold,
    ce_high_length=ce_high_length,
)
# k.use_remote("https://garygigabytes.com/kraken/parquet")  # no trailing slash
# df_multi = k.get_ohlcv_df_remote(symbols, interval=interval)
df_multi = k.get_ohlcv_df(symbols, interval=interval)

print(df_multi.head())

# remove multi-index
print(start)
print(end)

df_multi.columns = df_multi.columns.droplevel(0)
df = df_multi.loc[start:end]

print(f"\nMultiIndex Removed")
print(df.info())
print(df.head())
print(df.tail())

                                    BTC                              \
                                   open          high           low   
2023-01-01 00:00:00+00:00  16528.699219  16530.000000  16505.199219   
2023-01-01 04:00:00+00:00  16519.300781  16542.400391  16506.900391   
2023-01-01 08:00:00+00:00  16512.400391  16538.400391  16490.000000   
2023-01-01 12:00:00+00:00  16500.199219  16550.000000  16497.699219   
2023-01-01 16:00:00+00:00  16550.000000  16573.099609  16531.199219   

                                                                         
                                  close        volume trades base quote  
2023-01-01 00:00:00+00:00  16519.300781  2.636628e+10    803  BTC   USD  
2023-01-01 04:00:00+00:00  16510.699219  1.122105e+11   2919  BTC   USD  
2023-01-01 08:00:00+00:00  16500.199219  1.014384e+11   2536  BTC   USD  
2023-01-01 12:00:00+00:00  16549.900391  2.684441e+10   2002  BTC   USD  
2023-01-01 16:00:00+00:00  16553.699219  3.952236e+10   24

In [14]:
# Calculate indicators
signals = s.calc_signals(df.copy())

print(f"\nSignals:")
print(signals.info())
print(signals.head())
print(signals.tail())




Signals:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3241 entries, 2024-04-08 00:00:00+00:00 to 2025-09-30 00:00:00+00:00
Freq: 4h
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sar             1668 non-null   float64
 1   sar_s           3241 non-null   bool   
 2   adx             3214 non-null   float64
 3   adx_d           3212 non-null   float64
 4   adx_dmp         3228 non-null   float64
 5   adx_dmn         3228 non-null   float64
 6   adx_s           3241 non-null   bool   
 7   adx_bullish     3241 non-null   bool   
 8   entry_signal    3241 non-null   bool   
 9   open            3241 non-null   float32
 10  high            3241 non-null   float32
 11  low             3241 non-null   float32
 12  close           3241 non-null   float32
 13  volume          3241 non-null   float64
 14  trades          3241 non-null   Int64  
 15  base            3241 non-null   object 
 16  quote      

In [15]:
# Backtest
bt = Backtest(signals, interval=interval, transaction_fee=transaction_fee)
stats, profit_df = bt.run()


print(f"\nBacktest Stats:")
print(tabulate(stats.items(), tablefmt='github'))



Backtest Stats:
|------------------|------------|
| trading_days     |  540       |
| total_profit     | 2135.93    |
| total_profit_pct |  213.593   |
| sharpe           |    3.14493 |
| sortino          |    3.25285 |
| wins             |   27       |
| losses           |   12       |
| win_rate         |   69.2308  |
